# Days 6-7: Evaluation + Metadata Reranking

**Goal**:
- Build evaluation set (200 queries)
- Compute Recall@K, mAP, nDCG for visual-only retrieval
- Implement metadata reranking (category + color)
- Compare and save results

**Done checkpoint**:
- Metrics table printed for visual-only vs reranked
- `results/ablation.csv` saved to Drive

## 1. Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/Projects/VisualSearchSystem'
LOCAL_DATA_DIR = '/content/data'

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'CWD: {os.getcwd()}')

In [ ]:
import zipfile
from pathlib import Path

# Images must be on local disk — query paths in images.csv point to /content/data/raw/...
LOCAL_RAW_DIR = Path(LOCAL_DATA_DIR) / 'raw'
LOCAL_RAW_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR = LOCAL_RAW_DIR / 'images'

if IMAGES_DIR.exists() and any(IMAGES_DIR.iterdir()):
    print(f'Images on local disk: {len(list(IMAGES_DIR.glob("*.jpg"))):,} files.')
else:
    drive_zips = list((Path(PROJECT_DIR) / 'data' / 'raw').glob('*.zip'))
    if drive_zips:
        print('Extracting dataset from Drive zip to local disk...')
        with zipfile.ZipFile(drive_zips[0], 'r') as zf:
            zf.extractall(LOCAL_RAW_DIR)
        print(f'Done. {len(list(IMAGES_DIR.glob("*.jpg"))):,} images ready.')
    else:
        print('No zip found on Drive. Run Day 1 notebook first.')

In [ ]:
!pip install -q torch torchvision transformers faiss-cpu Pillow pandas numpy tqdm pyyaml scipy scikit-learn
print('Done.')

## 2. Load Config

In [ ]:
import yaml
from pathlib import Path

with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

BASE  = Path(PROJECT_DIR)
LOCAL = Path(LOCAL_DATA_DIR)

config['paths']['raw_images']      = str(LOCAL / 'raw')
config['paths']['metadata_csv']    = str(BASE / 'data/metadata/images.csv')
config['paths']['embeddings_path'] = str(BASE / 'data/embeddings/embeddings.npy')
config['paths']['id_map_path']     = str(BASE / 'data/embeddings/id_map.json')
config['paths']['index_path']      = str(BASE / 'data/embeddings/faiss.index')
config['paths']['eval_dir']        = str(BASE / 'data/eval')
config['paths']['results_dir']     = str(BASE / 'results')
config['model']['device']          = 'auto'
config['evaluation']['eval_set_size'] = 200
config['evaluation']['k_values']   = [5, 10, 20]

Path(config['paths']['results_dir']).mkdir(parents=True, exist_ok=True)
print('Config ready.')

## 3. Build Evaluation Set

In [ ]:
from src.evaluation.eval_set import build_eval_set, load_eval_set
from pathlib import Path
import os

eval_path = Path(config['paths']['eval_dir']) / 'eval_queries.csv'

# Delete old eval set if it was built with category-level relevance (too broad)
if eval_path.exists():
    import pandas as pd
    old_df = pd.read_csv(eval_path)
    if 'article_type' not in old_daf.columns:
        print('Old eval set found (category-level relevance). Rebuilding with article_type...')
        os.remove(eval_path)

if not eval_path.exists():
    print('Building eval set...')
    build_eval_set(config, n_queries=200)

eval_queries = load_eval_set(config)
print(f'\nLoaded {len(eval_queries)} eval queries.')
print(f'Sample query: ID={eval_queries[0]["query_image_id"]}, '
      f'article_type={eval_queries[0].get("article_type")}, '
      f'n_relevant={len(eval_queries[0]["relevant"])}')

## 4. Initialize Searcher + Rerankers

In [ ]:
from src.retrieval.search import Searcher
from src.reranking.metadata_reranker import MetadataReranker

searcher = Searcher(config)
metadata_reranker = MetadataReranker(config)

print(f'Searcher ready: {searcher.index.index.ntotal:,} vectors')

## 5. Run Evaluation: Visual-Only vs Metadata-Reranked

In [ ]:
from src.evaluation.metrics import compute_metrics, print_metrics_table
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

k_values = config['evaluation']['k_values']
candidate_pool = config['retrieval']['candidate_pool']

meta_df = pd.read_csv(config['paths']['metadata_csv'])
meta_df['image_id'] = meta_df['image_id'].astype(int)
meta_index = meta_df.set_index('image_id')

visual_evals = []
reranked_evals = []

for q in tqdm(eval_queries, desc='Evaluating'):
    qid = q['query_image_id']
    relevant = q['relevant']

    if qid not in meta_index.index:
        continue
    query_path = meta_index.loc[qid].get('path')
    if not query_path or not Path(query_path).exists():
        continue

    try:
        raw = searcher.search(query_path, k=candidate_pool)
    except Exception as e:
        continue

    # Visual-only
    visual_ids = [r['image_id'] for r in raw[:max(k_values)]]
    visual_evals.append({'retrieved': visual_ids, 'relevant': relevant})

    # Metadata reranked
    query_meta = meta_index.loc[qid].to_dict()
    reranked = metadata_reranker.rerank(raw, query_meta=query_meta, top_k=max(k_values))
    reranked_ids = [r['image_id'] for r in reranked]
    reranked_evals.append({'retrieved': reranked_ids, 'relevant': relevant})

print(f'Evaluated {len(visual_evals)} queries.')

## 6. Compute and Print Metrics

In [ ]:
from src.evaluation.metrics import compute_metrics, print_metrics_table

visual_metrics = compute_metrics(visual_evals, k_values=k_values)
reranked_metrics = compute_metrics(reranked_evals, k_values=k_values)

print_metrics_table(visual_metrics, label='Visual Only')
print_metrics_table(reranked_metrics, label='Metadata Reranked')

## 7. Visualize Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics_to_plot = ['Recall@10', 'Precision@10', 'mAP', 'nDCG@10']
x = np.arange(len(metrics_to_plot))
width = 0.35

visual_vals = [visual_metrics.get(m, 0) for m in metrics_to_plot]
reranked_vals = [reranked_metrics.get(m, 0) for m in metrics_to_plot]

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, visual_vals, width, label='Visual Only', color='steelblue')
bars2 = ax.bar(x + width/2, reranked_vals, width, label='Metadata Reranked', color='coral')

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Retrieval Evaluation: Visual vs Metadata Reranked')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.legend()
ax.set_ylim(0, 1.0)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/day6_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 8. Save Ablation Results

In [ ]:
import csv
from pathlib import Path

results_dir = Path(config['paths']['results_dir'])
results_dir.mkdir(parents=True, exist_ok=True)
output_csv = results_dir / 'ablation.csv'

all_results = {
    'visual_only': visual_metrics,
    'metadata_reranked': reranked_metrics,
}

fieldnames = ['mode'] + sorted(visual_metrics.keys())
with open(output_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for mode_name, metrics in all_results.items():
        writer.writerow({'mode': mode_name, **metrics})

print(f'Ablation results saved to {output_csv}')

# Display as DataFrame
import pandas as pd
pd.read_csv(output_csv)

## ✅ Days 6-7 Done Checkpoint

In [ ]:
from pathlib import Path

assert Path(config['paths']['eval_dir'], 'eval_queries.csv').exists()
assert Path(config['paths']['results_dir'], 'ablation.csv').exists()
assert 'mAP' in visual_metrics

print('Days 6-7 COMPLETE')
print(f'  Eval queries:     {len(eval_queries)}')
print(f'  Visual mAP:       {visual_metrics["mAP"]:.4f}')
print(f'  Reranked mAP:     {reranked_metrics["mAP"]:.4f}')
improvement = reranked_metrics['mAP'] - visual_metrics['mAP']
print(f'  mAP improvement:  {improvement:+.4f}')